# 11 · MongoDB, live

Short notebook, and every cell writes to the **real** collection the pipeline
reads.

You insert a document. You run the pipeline. It lands. Then you insert one where
a field has **moved**, and you watch the contract deal with it in front of the
room.

In notebook 4 you found a moved field in data somebody else generated. Here you
are the mobile team.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import datetime as dt, json, random
import psycopg
from pymongo import MongoClient
from pipelines.lib.config import dsn, SCHEMA, MONGO_URI

collection = MongoClient(MONGO_URI).kerb_app.driver_app_events
print(f'{collection.estimated_document_count():,} documents in the collection')

---

## Step 1 · Send one document, the way the app sends it today

Release 4.1.5. Surge at `payload.surge_multiplier`.

In [ ]:
def send(surge_path, version, trip_id=None):
    """Insert one driver app document, putting surge wherever you say."""
    trip_id = trip_id or f'TRP-NB11-{random.randint(10000, 99999)}'
    doc = {
        'event_id':   f'EV-NB11-{random.randint(100000, 999999)}',
        'trip_id':    trip_id,
        'driver_id':  f'DRV{random.randint(1, 2800):06d}',
        'event_type': 'trip_offer',
        'ts':         dt.datetime.now(dt.timezone.utc).replace(microsecond=0, tzinfo=None),
        'app':        {'version': version, 'platform': 'ios', 'build': 415},
        'payload':    {'eta_s': 240, 'distance_km': 3.1},
    }

    # walk the path, creating each level, and put the value at the end
    cur, surge = doc, round(random.uniform(1.0, 2.4), 2)
    for key in surge_path[:-1]:
        cur = cur.setdefault(key, {})
    cur[surge_path[-1]] = surge

    collection.insert_one(dict(doc))
    print(f"sent {doc['event_id']}  app {version}  surge {surge} at {'.'.join(surge_path)}")
    return doc['event_id'], trip_id

good_id, good_trip = send(('payload', 'surge_multiplier'), '4.1.5')

## Run the pipeline and watch it land

In [ ]:
run('-m', 'pipelines.p3_bronze_driver_app')

In [ ]:
sql(f"""
    SELECT event_id, trip_id, app_version, surge, happened_at
    FROM {SCHEMA}.bronze_driver_app
    WHERE event_id LIKE 'EV-NB11-%'
    ORDER BY happened_at DESC LIMIT 5
""", 'documents you sent, in the warehouse')

---

## Step 2 · Now be the mobile team

Ship release 5.0, and move the field somewhere nobody agreed on.

The pipeline knows four paths:

```
payload.surge_multiplier
payload.pricing.surgeFactor
pricing.surge_multiplier
surge_multiplier
```

Put it at a fifth.

In [ ]:
moved_id, moved_trip = send(('payload', 'fare', 'multiplier'), '5.0.0')

In [ ]:
run('-m', 'pipelines.p3_bronze_driver_app')

### Read that run line carefully

`rows_in` is larger than `rows_out`. **Nothing failed.** The pipeline read the
document perfectly well and refused to publish it, because it could not find a
value it was asked to carry.

## Where did it go?

In [ ]:
landed = fetch(f"""
    SELECT count(*) AS n FROM {SCHEMA}.bronze_driver_app WHERE event_id = '{moved_id}'
""").n[0]

print(f'rows in the warehouse for {moved_id}: {landed}\n')

sql(f"""
    SELECT reason, payload
    FROM {SCHEMA}.quarantine
    WHERE pipeline = 'p3_bronze_driver_app'
      AND payload::text LIKE '%{moved_trip}%'
    LIMIT 1
""", 'held, with the reason and the original document')

**Zero rows landed. One record held, with the whole document attached.**

That is the difference between *"surge looks low this week"* and *"here is the
document, here is the release that sent it, here is the path it used"*.

---

## Step 3 · The fix is one line

The contract is a tuple. Add the path to it.

Here it is done live, in this kernel, so you can watch the held record land
without leaving the notebook.

In [ ]:
import pipelines.p3_bronze_driver_app as p3

print('before:')
for path in p3.SURGE_PATHS:
    print('   ', '.'.join(path))

p3.SURGE_PATHS = p3.SURGE_PATHS + (('payload', 'fare', 'multiplier'),)   # release 5.0

print('\nafter:')
for path in p3.SURGE_PATHS:
    print('   ', '.'.join(path))

In [ ]:
p3.run()

**No migration. No backfill script. No redeploy.** One tuple grew by one line
and the record that could not be published now can be.

To make it permanent, add the same line to `SURGE_PATHS` in
`pipelines/p3_bronze_driver_app.py`:

```python
SURGE_PATHS = (
    ("payload", "surge_multiplier"),
    ("payload", "pricing", "surgeFactor"),
    ("payload", "fare", "multiplier"),        # <- release 5.0
    ("pricing", "surge_multiplier"),
    ("surge_multiplier",),
)
```

In [ ]:
sql(f"""
    SELECT event_id, app_version, surge
    FROM {SCHEMA}.bronze_driver_app
    WHERE event_id IN ('{good_id}', '{moved_id}')
""", 'both documents, once the contract knows the new path')

---

## Step 4 · What the shape of the collection looks like now

In [ ]:
report = collection.aggregate([
    {'$match': {'event_type': 'trip_offer'}},
    {'$project': {
        'at_payload': {'$cond': [{'$ifNull': ['$payload.surge_multiplier', False]}, 1, 0]},
        'at_nested':  {'$cond': [{'$ifNull': ['$payload.pricing.surgeFactor', False]}, 1, 0]},
        'at_fare':    {'$cond': [{'$ifNull': ['$payload.fare.multiplier', False]}, 1, 0]},
        'at_pricing': {'$cond': [{'$ifNull': ['$pricing.surge_multiplier', False]}, 1, 0]},
    }},
    {'$group': {'_id': None,
                'at_payload': {'$sum': '$at_payload'},
                'at_nested':  {'$sum': '$at_nested'},
                'at_fare':    {'$sum': '$at_fare'},
                'at_pricing': {'$sum': '$at_pricing'},
                'documents':  {'$sum': 1}}},
]).next()

NAMES = {'at_payload': 'payload.surge_multiplier',
         'at_nested':  'payload.pricing.surgeFactor',
         'at_fare':    'payload.fare.multiplier',
         'at_pricing': 'pricing.surge_multiplier'}

total = report['documents']
print(f'{total:,} trip_offer documents, and the surge value lives at:\n')
for key, label in NAMES.items():
    n_found = report[key]
    print(f'  {label:30} {n_found:>8,}   {n_found / total:>7.3%}')

**Four different shapes in one collection, all of them real, none of them
wrong.** That is what a schemaless store looks like after two years.

The contract is the only thing standing between that and a column full of nulls
nobody can explain.

---

## Tidy up

The documents you sent stay in Mongo, which is honest: you really did send them.
Remove them if you want a clean collection for the next class.

In [ ]:
deleted = collection.delete_many({'event_id': {'$regex': '^EV-NB11-'}}).deleted_count
print(f'removed {deleted} documents you sent')

with psycopg.connect(dsn(), autocommit=True) as c:
    n = c.execute(f"DELETE FROM {SCHEMA}.bronze_driver_app WHERE event_id LIKE 'EV-NB11-%'").rowcount
print(f'removed {n} rows from the warehouse')

---

## What you learned

- A document store lets **every release invent its own shape**, and nothing complains
- A moved field is **not an error**. The pipeline reads the document fine
- The contract turns an invisible problem into **a held record with the document
  attached**
- Fixing it is **one line**, with no migration and no backfill
- Ask the collection what shapes it holds. The answer is rarely one